# 5 · Physical and cyber mitigations in one currency

Each option alone, the two Pareto fronts, and one-at-a-time sensitivity. The 500-draw
uncertainty study is in `validation/mitigation.py`. See
[Lesson 5](../docs/lessons/05-comparing-mitigations.md).

In [1]:
import matplotlib.pyplot as plt

from cpsrbd import plotting

plotting.use_style()

In [2]:
from cpsrbd.consequence_driven_analysis import navigation_case_study
from cpsrbd.mitigation_tradeoff_analysis import (
    assess,
    case_study_mitigations,
    pareto_front,
    portfolios,
    tornado,
)

model = navigation_case_study()
options = case_study_mitigations()
for a in sorted(assess(model, options), key=lambda a: -a.risk_reduction):
    print(f"{a.name:28s} invest {a.investment:4.0f}  risk -{a.risk_reduction:5.1%}  cheapest attack {a.cost_after:g}")

redundant_mission_computer   invest  300  risk -46.7%  cheapest attack 5
stronger_integrity_monitor   invest  150  risk -28.0%  cheapest attack 5
anti_spoof_gnss_antenna      invest  250  risk -28.0%  cheapest attack 5
offline_software_loader      invest  120  risk -18.6%  cheapest attack 42
bus_guardian                 invest  180  risk -13.4%  cheapest attack 9
harden_maintenance_laptop    invest   40  risk -13.0%  cheapest attack 9
authenticate_bus_messages    invest  200  risk - 5.6%  cheapest attack 5
independent_nav_power        invest   80  risk - 0.1%  cheapest attack 5


In [3]:
every = portfolios(model, options)
for objective in ("risk", "effort"):
    print(f"Pareto front ({objective}):")
    for p in pareto_front(every, objective)[:6]:
        print(f"  {p.investment:5.0f}  P {p.q:.2e}  attack {p.attacker_cost:g}  {', '.join(p.names) or '(none)'}")

Pareto front (risk):
      0  P 1.02e-03  attack 5  (none)
     40  P 8.83e-04  attack 9  harden_maintenance_laptop
    120  P 8.27e-04  attack 42  offline_software_loader
    150  P 7.31e-04  attack 5  stronger_integrity_monitor
    190  P 5.99e-04  attack 9  stronger_integrity_monitor, harden_maintenance_laptop
    270  P 5.42e-04  attack 42  stronger_integrity_monitor, offline_software_loader
Pareto front (effort):
      0  P 1.02e-03  attack 5  (none)
     40  P 8.83e-04  attack 9  harden_maintenance_laptop
    120  P 8.27e-04  attack 42  offline_software_loader
    300  P 8.23e-04  attack 46  bus_guardian, offline_software_loader
    500  P 8.21e-04  attack 52  authenticate_bus_messages, bus_guardian, offline_software_loader


In [4]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter([p.investment for p in every], [p.q for p in every], s=10, c=[p.physical_share for p in every], cmap="coolwarm_r")
front = pareto_front(every, "risk")
ax.step([p.investment for p in front], [p.q for p in front], where="post", color="black")
ax.set_yscale("log")
ax.set_xlabel("investment")
ax.set_ylabel("P(consequence)")
ax.set_title("256 portfolios, colour = physical share")

Text(0.5, 1.0, '256 portfolios, colour = physical share')

In [5]:
for b in tornado(model)[:6]:
    print(f"{b.event:30s} {b.q_low:.2e} .. {b.q_high:.2e}")

mc_hw                          6.82e-04 .. 2.01e-03
gnss_spoof                     8.04e-04 .. 1.65e-03
integrity_monitor_miss         8.04e-04 .. 1.65e-03
compromise_maintenance_laptop  8.90e-04 .. 1.39e-03
flood_bus                      9.24e-04 .. 1.29e-03
inject_false_nav_messages      9.77e-04 .. 1.13e-03
